In [34]:
airport_codes = [
    "LHR",  # London Heathrow (UK) - Major European Hub
    "JFK",  # John F. Kennedy, New York (USA) - Primary US Gateway
    "DXB",  # Dubai International (UAE) - Massive Middle East Hub
    "HND",  # Haneda, Tokyo (Japan) - Busy Asian Hub
    "SIN",  # Changi (Singapore) - Top-rated International Hub
    "CDG",  # Charles de Gaulle, Paris (France) - European Gateway
    "LAX",  # Los Angeles (USA) - Major US West Coast Hub
    "SYD",  # Kingsford Smith, Sydney (Australia) - Oceania Hub
    "GRU",  # São Paulo/Guarulhos (Brazil) - Busiest in South America
    "JNB",  # OR Tambo, Johannesburg (South Africa) - Major African Hub
    "DEL",  # Indira Gandhi, Delhi (India) - Major South Asian Hub
    "MLE",  # Velana, Malé (Maldives) - Island Resort Destination
    "HNL",  # Honolulu, Hawaii (USA) - Pacific Island Hub
    "AMS",  # Schiphol, Amsterdam (Netherlands) - Massive European Transit Hub
    "IST",  # Istanbul (Turkey) - Bridge between Europe and Asia
]

In [ ]:
#Data extraction from Aerodata box API's
import http.client
import json
import pandas as pd
import time

Airport_dfs=[]

def get_airport_details(airport_code):
    conn = http.client.HTTPSConnection("aerodatabox.p.rapidapi.com")
    headers = {
    'x-rapidapi-key': "api_key",
    'x-rapidapi-host': "aerodatabox.p.rapidapi.com",
    'Content-Type': "application/json"
    }
    conn.request("GET", f"/airports/iata/{airport_code}", headers=headers)
    res = conn.getresponse()
    data = res.read()
    json_text = data.decode("utf-8")
    data_dict = json.loads(json_text)
    df = pd.json_normalize(data_dict)
# 2. fields that are required for airport table
    columns_to_keep = [
    "icao",
    "iata",
    "shortName",
    "fullName",
    "timeZone",
    "municipalityName",
    "location.lat",
    "location.lon",
    "country.code",
    "country.name",
    "continent.code",
    "continent.name"
]

    df_filtered = df[columns_to_keep]
    return df_filtered

# Collect all individual DataFrames into the list
for code in airport_codes:
    time.sleep(1.5)  # Add a 1-second delay between requests-aerodata api is not allowing more than one request per second
    try:
        airport_df = get_airport_details(code)
        Airport_dfs.append(airport_df)
    except Exception as e:
        print(f"Failed to fetch data for {code}: {e}")
# Combine everything into a single master DataFrame at once
master_df = pd.concat(Airport_dfs, ignore_index=True)
# Saves without the row index numbers
print(master_df)
print(master_df.shape)


    icao iata                                    shortName  \
0   EGLL  LHR                                     Heathrow   
1   KJFK  JFK                               John F Kennedy   
2   OMDB  DXB                                        Dubai   
3   RJTT  HND                                       Haneda   
4   WSSS  SIN                                       Changi   
5   LFPG  CDG                            Charles de Gaulle   
6   KLAX  LAX                                  Los Angeles   
7   YSSY  SYD                              Kingsford Smith   
8   SBGR  GRU  Guarulhos - Governador André Franco Montoro   
9   FAOR  JNB                        Johannesburg OR Tambo   
10  VIDP  DEL                                Indira Gandhi   
11  VRMM  MLE                                         Malé   
12  PHNL  HNL                                     Honolulu   
13  EHAM  AMS                                     Schiphol   
14  LTFM  IST                                     Istanbul   

       

In [ ]:
# Saves without the row index numbers
master_df.to_csv("airports.csv", index=False)


In [ ]:
# Data extraction from AeroDataBox Flights API

import http.client
import json
import pandas as pd
import time

# Columns required for the Flights table
flight_columns_to_keep = [
    "number",                          # flight_number
    "aircraft.reg",                    # aircraft_registration
    "origin_iata",
    "destination_iata",
    "departure.scheduledTime.utc",     # scheduled_departure
    "departure.revisedTime.utc",       # actual_departure
    "arrival.scheduledTime.utc",       # scheduled_arrival
    "arrival.revisedTime.utc",         # actual_arrival
    "status",
    "airline.iata"                     # airline_code
]

# Rename API columns to project sql schema columns
column_mapping = {
    "number": "flight_number",
    "aircraft.reg": "aircraft_registration",
    "departure.scheduledTime.utc": "scheduled_departure",
    "departure.revisedTime.utc": "actual_departure",
    "arrival.scheduledTime.utc": "scheduled_arrival",
    "arrival.revisedTime.utc": "actual_arrival",
    "airline.iata": "airline_code"
}

flights_dfs = []

def get_flight_details(airport_code):

    conn = http.client.HTTPSConnection("aerodatabox.p.rapidapi.com")

    headers = {
        'x-rapidapi-key': "api_key",
        'x-rapidapi-host': "aerodatabox.p.rapidapi.com",
        'Content-Type': "application/json"
    }
    conn.request(
        "GET",
        f"/flights/airports/iata/{airport_code}?offsetMinutes=-120&durationMinutes=720&withLeg=true&direction=Both&withCancelled=true&withCodeshared=true&withCargo=true&withPrivate=true&withLocation=false",
        headers=headers
    )
    res = conn.getresponse()
    data = res.read()
    json_text = data.decode("utf-8")
    data_dict = json.loads(json_text)

    # Normalize departures and arrivals separately because of the nested nature of the keys
    df_departures = pd.json_normalize(data_dict["departures"])
    df_arrivals = pd.json_normalize(data_dict["arrivals"])

    # Create origin and destination columns

    # Departures
    df_departures["origin_iata"] = airport_code
    df_departures["destination_iata"] = df_departures["arrival.airport.iata"]

    # Arrivals
    df_arrivals["origin_iata"] = df_arrivals["departure.airport.iata"]
    df_arrivals["destination_iata"] = airport_code

    # Combine departures and arrivals
    df = pd.concat(
        [df_departures, df_arrivals],
        ignore_index=True
    )
    # Keep required columns only
    available_columns = [
        col for col in flight_columns_to_keep
        if col in df.columns
    ]
    df_filtered = df[available_columns]

    # Rename columns to match project schema
    df_renamed = df_filtered.rename(columns=column_mapping)

    # Create flight_id (TEXT PRIMARY KEY)
    df_renamed["flight_id"] = (
        df_renamed["flight_number"]
        .str.replace(" ", "", regex=False)
        + "_"
        + df_renamed["scheduled_departure"].astype(str)
    )

    return df_renamed

# Fetch flights for all selected airports
for code in airport_codes:
    time.sleep(1.5)  # AeroDataBox rate limit
    try:
        flight_df = get_flight_details(code)
        flights_dfs.append(flight_df)
    except Exception as e:
        print(f"Failed to fetch data for {code}: {e}")

# Combine all airports into one master dataframe
master_flight_df = pd.concat(
    flights_dfs,
    ignore_index=True
)

print(master_flight_df.head())
print("\nShape:", master_flight_df.shape)
print("\nColumns:")
print(master_flight_df.columns.tolist())

# Save snapshot
master_flight_df.to_csv(
    "flights_master.csv",
    index=False
)


  flight_number aircraft_registration origin_iata destination_iata  \
0        BA 348                   NaN         LHR              MRS   
1       QR 6219                   NaN         LHR              MRS   
2        BA 598                   NaN         LHR              VCE   
3       QR 8337                   NaN         LHR              VCE   
4       AA 6897                   NaN         LHR              VCE   

  scheduled_departure   actual_departure  scheduled_arrival  \
0   2026-05-31 05:05Z  2026-05-31 05:11Z  2026-05-31 07:00Z   
1   2026-05-31 05:05Z  2026-05-31 05:11Z  2026-05-31 07:00Z   
2   2026-05-31 05:05Z  2026-05-31 05:15Z  2026-05-31 07:20Z   
3   2026-05-31 05:05Z  2026-05-31 05:15Z  2026-05-31 07:20Z   
4   2026-05-31 05:05Z  2026-05-31 05:15Z  2026-05-31 07:20Z   

      actual_arrival    status airline_code                 flight_id  
0  2026-05-31 06:34Z  Departed           BA   BA348_2026-05-31 05:05Z  
1  2026-05-31 06:34Z  Departed           QR  QR6219_2026

In [ ]:
unique_aircraft = (
    master_flight_df["aircraft_registration"]
    .dropna()
    .unique()
)
print(unique_aircraft)
print(len(unique_aircraft))

['OE-LZQ' 'YL-ABU' 'G-TTNJ' ... 'TC-LSK' 'TC-LLA' 'TC-JSU']
1851


In [3]:
import pandas as pd
airports_df = pd.read_csv("airports.csv")
airports_df = airports_df.rename(columns={
    "municipalityName": "city",
    "location.lat": "latitude",
    "location.lon": "longitude",
    "country.code": "country_code",
    "country.name": "country_name",
    "continent.code": "continent_code",
    "continent.name": "continent_name"
})
airports_df.to_csv("airports_clean.csv", index=False)

In [2]:
# find the aircraft codes in flights csv with high occurance
import pandas as pd
flights_df = pd.read_csv("flights_master.csv")
top_aircraft_codes = (
    flights_df["aircraft_registration"]
    .dropna()
    .value_counts()
    .head(100)
    .index
    .tolist()
)
print(top_aircraft_codes[:5])

['PH-BXN', 'PH-AXM', 'PH-NXX', 'PH-NXN', 'PH-BXP']


In [3]:
import http.client
import json
import pandas as pd
import time

aircraft_columns_to_keep = [
    "reg",
    "typeName",
    "iataCodeShort",
    "airlineName"
]

column_mapping = {
    "reg": "registration",
    "typeName": "model",
    "iataCodeShort": "icao_type_code",
    "airlineName": "owner"
}

aircraft_dfs = []

def get_aircraft_details(registration):
    conn = http.client.HTTPSConnection("aerodatabox.p.rapidapi.com")
    headers = {
        "x-rapidapi-key": "api_key",
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }
    conn.request(
        "GET",
        f"/aircrafts/reg/{registration}/all",
        headers=headers
    )
    res = conn.getresponse()
    data = res.read()
    # Debug (remove later)
    print(f"{registration} -> Status: {res.status}")
    json_text = data.decode("utf-8")

    # Empty response
    if not json_text.strip():
        print(f"Empty response for {registration}")
        return None
    try:
        data_dict = json.loads(json_text)
    except Exception:
        print(f"Invalid JSON for {registration}")
        return None
    # API sometimes returns []
    if not data_dict:
        print(f"No aircraft data for {registration}")
        return None
    df = pd.DataFrame(data_dict)
    # Keep only columns that exist
    available_cols = [
        col for col in aircraft_columns_to_keep
        if col in df.columns
    ]
    if len(available_cols) == 0:
        print(f"No expected columns found for {registration}")
        return None
    df = df[available_cols]
    df = df.rename(columns=column_mapping)
    # Create manufacturer from model
    if "model" in df.columns:
        df["manufacturer"] = (
            df["model"]
            .astype(str)
            .str.split()
            .str[0]
        )
    else:
        df["manufacturer"] = None
    return df

for reg in top_aircraft_codes[:100]:

    time.sleep(1.5)

    try:
        aircraft_df = get_aircraft_details(reg)

        if aircraft_df is not None:
            aircraft_dfs.append(aircraft_df)

    except Exception as e:
        print(f"Failed to fetch data for {reg}: {e}")


# Combine results
if aircraft_dfs:

    master_aircraft_df = pd.concat(
        aircraft_dfs,
        ignore_index=True
    )

    print(master_aircraft_df.head())
    print(master_aircraft_df.shape)

else:
    print("No aircraft data collected.")

PH-BXN -> Status: 200
PH-AXM -> Status: 200
PH-NXX -> Status: 204
Empty response for PH-NXX
PH-NXN -> Status: 200
PH-BXP -> Status: 200
PH-BCK -> Status: 200
PH-BXU -> Status: 200
PH-EXF -> Status: 200
PH-EXZ -> Status: 200
PH-AXE -> Status: 200
PH-NXF -> Status: 200
F-HOZC -> Status: 200
PH-NXV -> Status: 200
PH-AXR -> Status: 200
PH-NXR -> Status: 200
PH-AXG -> Status: 200
PH-BXV -> Status: 200
PH-BXH -> Status: 200
PH-BGG -> Status: 200
PH-BXI -> Status: 200
PH-BXM -> Status: 200
PH-BXO -> Status: 200
PH-EXM -> Status: 200
PH-AXH -> Status: 200
PH-AXA -> Status: 200
PH-EZM -> Status: 200
PH-NXY -> Status: 200
PH-AXI -> Status: 200
PH-BGH -> Status: 200
PH-BCB -> Status: 200
F-GKXC -> Status: 200
PH-AXF -> Status: 200
D-AKJC -> Status: 200
PH-BGM -> Status: 200
PH-AXC -> Status: 200
PH-EXW -> Status: 200
PH-BXD -> Status: 204
Empty response for PH-BXD
PH-NXG -> Status: 200
PH-EZF -> Status: 200
PH-EXS -> Status: 200
PH-BXF -> Status: 200
PH-BXC -> Status: 200
PH-BCL -> Status: 200
PH

In [4]:
master_aircraft_df = master_aircraft_df.drop_duplicates(
    subset=["registration"]
)

In [5]:
master_aircraft_df.to_csv("aircraft.csv", index=False)

In [60]:
# Data extraction from AeroDataBox API's
import http.client
import json
import pandas as pd
import time
import statistics

delay_dfs = []
DATE = "2026-05-31"
AIRPORTS_ICAO = pd.read_csv("airports_clean.csv").set_index("iata")["icao"].to_dict()

def fetch(icao, from_dt, to_dt):
    try:
        conn = http.client.HTTPSConnection("aerodatabox.p.rapidapi.com")
        headers = {
            'x-rapidapi-key':  "api_key",
            'x-rapidapi-host': "aerodatabox.p.rapidapi.com",
            'Content-Type':    "application/json"
        }
        conn.request("GET", f"/airports/icao/{icao}/delays/{from_dt}/{to_dt}", headers=headers)
        raw  = conn.getresponse().read().decode("utf-8")
        data = json.loads(raw)
        return data if isinstance(data, list) else []
    except Exception as e:
        print(f"Skipping {icao}: {e}")
        return []

def to_minutes(t):
    p = t.split(":")
    return int(p[0]) * 60 + int(p[1])

def get_airport_delays(iata, icao, date):
    slots = fetch(icao, f"{date}T00:00", f"{date}T12:00") + \
            fetch(icao, f"{date}T12:00", f"{date}T23:59")
    if not slots:
        return None

    total, cancelled, delayed, mins = 0, 0, 0, []
    for s in slots:
        for key in ["departuresDelayInformation", "arrivalsDelayInformation"]:
            info = s.get(key, {})
            total     += info.get("numTotal", 0)
            cancelled += info.get("numCancelled", 0)
            if info.get("delayIndex", 0) > 1:
                delayed += info.get("numTotal", 0)
            mins.append(to_minutes(info.get("medianDelay", "00:00:00")))

    # Fields required for airport_delays table
    columns_to_keep = [
        "airport_iata", "delay_date", "numTotal",
        "numDelayed", "avgDelay", "medianDelay", "numCancelled"
    ]

    # Rename to match airport_delays table columns
    columns_mapping = {
        "numTotal":     "total_flights",
        "numDelayed":   "delayed_flights",
        "avgDelay":     "avg_delay_min",
        "medianDelay":  "median_delay_min",
        "numCancelled": "canceled_flights"
    }

    df = pd.DataFrame([{
        "airport_iata": iata,
        "delay_date":   date,
        "numTotal":     total,
        "numDelayed":   delayed,
        "avgDelay":     round(sum(mins) / len(mins), 2),
        "medianDelay":  round(statistics.median(mins), 2),
        "numCancelled": cancelled
    }])

    df_filtered = df[columns_to_keep]
    df_renamed  = df_filtered.rename(columns=columns_mapping)
    return df_renamed

# Collect all individual DataFrames into the list
for iata, icao in AIRPORTS_ICAO.items():
    time.sleep(1.5)  # Add a delay between requests - AeroDataBox API is not allowing more than one request per second
    try:
        delay_df = get_airport_delays(iata, icao, DATE)
        if delay_df is not None:
            delay_dfs.append(delay_df)
    except Exception as e:
        print(f"Failed to fetch data for {iata}: {e}")

# Combine everything into a single master DataFrame at once
delays_master_df = pd.concat(delay_dfs, ignore_index=True)
# Saves without the row index numbers
print(delays_master_df)
print(delays_master_df.shape)
delays_master_df.to_csv("airport_delays.csv", index=False)

Skipping VRMM: Expecting value: line 1 column 1 (char 0)
   airport_iata  delay_date  total_flights  delayed_flights  avg_delay_min  \
0           LHR  2026-05-31           3236              947           8.28   
1           JFK  2026-05-31           3756             1853          17.17   
2           DXB  2026-05-31           3667                0           5.54   
3           HND  2026-05-31           3402                0           5.47   
4           SIN  2026-05-31           3029                0           7.63   
5           CDG  2026-05-31           3420              877           7.11   
6           LAX  2026-05-31           4653               29          11.40   
7           SYD  2026-05-31           1745                0           2.52   
8           GRU  2026-05-31           2346                0           0.00   
9           JNB  2026-05-31            948              130           4.48   
10          DEL  2026-05-31           4999                0           0.00   
11     

In [7]:
flights_df = pd.read_csv("flights_master.csv")
print(flights_df.info())
print(flights_df.isnull().sum())
print(flights_df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21322 entries, 0 to 21321
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   flight_number          21322 non-null  object
 1   aircraft_registration  6056 non-null   object
 2   origin_iata            19893 non-null  object
 3   destination_iata       19941 non-null  object
 4   scheduled_departure    19725 non-null  object
 5   actual_departure       15644 non-null  object
 6   scheduled_arrival      19866 non-null  object
 7   actual_arrival         14931 non-null  object
 8   status                 21322 non-null  object
 9   airline_code           20961 non-null  object
 10  flight_id              21322 non-null  object
dtypes: object(11)
memory usage: 1.8+ MB
None
flight_number                0
aircraft_registration    15266
origin_iata               1429
destination_iata          1381
scheduled_departure       1597
actual_departure          5

In [8]:
#data cleaning and preprocessing
flights_df = pd.read_csv("flights_master.csv")

# Drop duplicates
flights_df = flights_df.drop_duplicates()

# Drop rows where critical columns are null
critical_columns = ["origin_iata", "destination_iata", "scheduled_departure", "scheduled_arrival"]
flights_df = flights_df.dropna(subset=critical_columns)

# Fill non-critical nulls with None (SQL friendly)
flights_df["aircraft_registration"] = flights_df["aircraft_registration"].fillna("Unknown")
flights_df["actual_departure"]      = flights_df["actual_departure"].fillna("Not Available")
flights_df["actual_arrival"]        = flights_df["actual_arrival"].fillna("Not Available")
flights_df["airline_code"]          = flights_df["airline_code"].fillna("Unknown")

print(flights_df.isnull().sum())
print(flights_df.shape)
flights_df.to_csv("flights_master.csv", index=False)

flight_number            0
aircraft_registration    0
origin_iata              0
destination_iata         0
scheduled_departure      0
actual_departure         0
scheduled_arrival        0
actual_arrival           0
status                   0
airline_code             0
flight_id                0
dtype: int64
(17640, 11)


In [9]:
aircraft_df = pd.read_csv("aircraft.csv")
print(aircraft_df.info())
print(aircraft_df.isnull().sum())
print(aircraft_df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98 entries, 0 to 97
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   registration    98 non-null     object
 1   model           98 non-null     object
 2   icao_type_code  94 non-null     object
 3   owner           98 non-null     object
 4   manufacturer    98 non-null     object
dtypes: object(5)
memory usage: 4.0+ KB
None
registration      0
model             0
icao_type_code    4
owner             0
manufacturer      0
dtype: int64
0


In [10]:

aircraft_df["icao_type_code"] = aircraft_df["icao_type_code"].fillna("Unknown")
print(aircraft_df.isnull().sum())
aircraft_df.to_csv("aircraft.csv", index=False)

registration      0
model             0
icao_type_code    0
owner             0
manufacturer      0
dtype: int64


In [11]:
delays_df = pd.read_csv("airport_delays.csv")
print(delays_df.info())
print(delays_df.isnull().sum())
print(delays_df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   airport_iata      14 non-null     object 
 1   delay_date        14 non-null     object 
 2   total_flights     14 non-null     int64  
 3   delayed_flights   14 non-null     int64  
 4   avg_delay_min     14 non-null     float64
 5   median_delay_min  14 non-null     float64
 6   canceled_flights  14 non-null     int64  
dtypes: float64(2), int64(3), object(2)
memory usage: 916.0+ bytes
None
airport_iata        0
delay_date          0
total_flights       0
delayed_flights     0
avg_delay_min       0
median_delay_min    0
canceled_flights    0
dtype: int64
0
